# YOLO11 to NCNN Conversion Guidelines
This notebook automates the conversion of YOLO11 models (Detection, Segmentation, Pose, OBB, and Classification) to NCNN format using PNNX, with support for dynamic shape inference.

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")
project = rf.workspace("dylans-workspace-3init").project("my-first-project-sbk3a")
version = project.version(1)
dataset = version.download("yolov11") # This downloads the images AND the data.yaml

loading Roboflow workspace...
loading Roboflow project...


## 1. Install Dependencies

In [2]:
!pip3 install -U ultralytics pnnx ncnn

# Train

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060


In [4]:
from ultralytics import YOLO

# 1. Load a pretrained YOLO11n model
model = YOLO('yolo11n.pt')

# 2. Train the model
# 'data.yaml' contains paths to your orange ball images and class names
model.train(data=f"{dataset.location}/data.yaml", epochs=100, imgsz=640)

# 3. Update the weights path for the conversion steps
# The best weights are saved in 'runs/detect/train/weights/best.pt'
my_model_path = 'runs/detect/train/weights/best.pt'

Ultralytics 8.4.21  Python-3.10.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\dylan\Downloads\tune\My-First-Project-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

## 2. Export YOLO11 TorchScript
We start by exporting the official weights to TorchScript format.

In [1]:
import shutil, os

# --- nano (n) ---
# Detection: export trained best.pt and rename to yolo11n.torchscript for pnnx
!yolo export model=C:\YOLO-android/yolo-mvp/runs/detect/train3/weights/best.pt format=torchscript
src_det = r'C:\YOLO-android\yolo-mvp\runs\detect\train3\weights\best.torchscript'
if os.path.exists(src_det):
    shutil.copy2(src_det, 'yolo11n.torchscript')
    print("Copied best.torchscript → yolo11n.torchscript")
else:
    print(f"WARNING: {src_det} not found — check your training output path")

# !yolo export model=yolo11n-seg.pt  format=torchscript
# !yolo export model=yolo11n-pose.pt format=torchscript
# !yolo export model=yolo11n-cls.pt  format=torchscript
# !yolo export model=yolo11n-obb.pt  format=torchscript

# # --- small (s) ---
# !yolo export model=yolo11s.pt      format=torchscript
# !yolo export model=yolo11s-seg.pt  format=torchscript
# !yolo export model=yolo11s-pose.pt format=torchscript
# !yolo export model=yolo11s-cls.pt  format=torchscript
# !yolo export model=yolo11s-obb.pt  format=torchscript

# # --- medium (m) ---
# !yolo export model=yolo11m.pt      format=torchscript
# !yolo export model=yolo11m-seg.pt  format=torchscript
# !yolo export model=yolo11m-pose.pt format=torchscript
# !yolo export model=yolo11m-cls.pt  format=torchscript
# !yolo export model=yolo11m-obb.pt  format=torchscript


Ultralytics 8.4.21  Python-3.10.10 torch-2.5.1+cu121 CPU (Intel Core i7-7700 3.60GHz)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'C:\YOLO-android\yolo-mvp\runs\detect\train3\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

TorchScript: starting export with torch 2.5.1+cu121...
TorchScript: export success  3.1s, saved as 'C:\YOLO-android\yolo-mvp\runs\detect\train3\weights\best.torchscript' (10.4 MB)

Export complete (3.6s)
Results saved to C:\YOLO-android\yolo-mvp\runs\detect\train3\weights
Predict:         yolo predict task=detect model=C:\YOLO-android\yolo-mvp\runs\detect\train3\weights\best.torchscript imgsz=640 
Validate:        yolo val task=detect model=C:\YOLO-android\yolo-mvp\runs\detect\train3\weights\best.torchscript imgsz=640 data=c:\Users\dylan\Downloads\tune\My-First-Project-1/data.yaml  
Visualize:       https://netron.app
 Learn more at https://docs.ultralyti

## 3. Initial PNNX Conversion (Static Shape)
Note: For classification models, this step is sufficient.

In [2]:
# --- nano (n): cls uses static shape (no patching needed) ---
!pnnx yolo11n.torchscript
# !pnnx yolo11n-seg.torchscript
# !pnnx yolo11n-pose.torchscript
# !pnnx yolo11n-cls.torchscript
# !pnnx yolo11n-obb.torchscript

# # --- small (s) ---
# !pnnx yolo11s.torchscript
# !pnnx yolo11s-seg.torchscript
# !pnnx yolo11s-pose.torchscript
# !pnnx yolo11s-cls.torchscript
# !pnnx yolo11s-obb.torchscript

# # --- medium (m) ---
# !pnnx yolo11m.torchscript
# !pnnx yolo11m-seg.torchscript
# !pnnx yolo11m-pose.torchscript
# !pnnx yolo11m-cls.torchscript
# !pnnx yolo11m-obb.torchscript


pnnxparam = yolo11n.pnnx.param
pnnxbin = yolo11n.pnnx.bin
pnnxpy = yolo11n_pnnx.py
pnnxonnx = yolo11n.pnnx.onnx
ncnnparam = yolo11n.ncnn.param
ncnnbin = yolo11n.ncnn.bin
ncnnpy = yolo11n_ncnn.py
fp16 = 1
optlevel = 2
device = cpu
inputshape = 
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2PSA
inline module = ultralytics.nn.modules.block.C3k
inline module = ultralytics.nn.modules.block.C3k2
inline module = ultralytics.nn.modules.block.DFL
inline module = ultralytics.nn.modules.block.PSABlock
inline module = ultralytics.nn.modules.block.SPPF
inline module = ultralytics.nn.modules.conv.Concat
inline module = ultralytics.nn.modules.conv.Conv
inline module = ultralytics.nn.modules.conv.DWConv
inline module

## 4. Automating Model Script Modifications
The following Python function automates the manual edits required for dynamic shape inference, including:
1. Modifying `reshape` to use `-1`.
2. Transposing tensors for faster processing.
3. Dropping the post-process part.
4. Fixing Area Attention for dynamic input sizes.

In [3]:
import os

def patch_yolo_script(filename, model_type='det'):
    if not os.path.exists(filename):
        print(f"File {filename} not found, skipping.")
        return

    with open(filename, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        # 1. Modify Area Attention (Universal fix for YOLO11)
        if 'v_96 = v_95.view(1, 2, 128, 1024)' in line:
            line = line.replace('1024', '-1')
        if 'v_106 = v_105.view(1, 128, 32, 32)' in line:
            line = "        v_106 = v_105.view(1, 128, v_95.size(2), v_95.size(3))\n"
        if 'v_107 = v_99.reshape(1, 128, 32, 32)' in line:
            line = "        v_107 = v_99.reshape(1, 128, v_95.size(2), v_95.size(3))\n"

        # 2. Handle Reshape/Transpose/Concat based on model type
        if '.view(1, ' in line and any(x in line for x in ['6400', '1600', '400', '16384', '4096', '1024']):
            line = line.replace('6400', '-1').replace('1600', '-1').replace('400', '-1')
            line = line.replace('16384', '-1').replace('4096', '-1').replace('1024', '-1')
            # rstrip (not strip) preserves leading indentation so Python syntax stays valid
            line = line.rstrip() + ".transpose(1, 2)\n"

        # 3. Adjust Concat axis (from dim=2 to dim=1 after transpose)
        if 'torch.cat' in line and 'dim=2' in line:
            line = line.replace('dim=2', 'dim=1')

        # 4. Modify Return statements (Dropping post-process)
        if 'return' in line and 'v_' in line:
            if model_type == 'det':   line = "        return v_238\n"
            elif model_type == 'seg': line = "        return v_264, v_215, v_196\n"
            elif model_type == 'pose':line = "        return v_257, v_208\n"
            elif model_type == 'obb': line = "        return v_259, v_208\n"

        new_lines.append(line)

    with open(filename, 'w') as f:
        f.writelines(new_lines)
    print(f"Patched {filename} successfully.")

# nano (n)
patch_yolo_script('yolo11n_pnnx.py',      'det')
# patch_yolo_script('yolo11n_seg_pnnx.py',  'seg')
# patch_yolo_script('yolo11n_pose_pnnx.py', 'pose')
# patch_yolo_script('yolo11n_obb_pnnx.py',  'obb')

# # small (s)
# patch_yolo_script('yolo11s_pnnx.py',      'det')
# patch_yolo_script('yolo11s_seg_pnnx.py',  'seg')
# patch_yolo_script('yolo11s_pose_pnnx.py', 'pose')
# patch_yolo_script('yolo11s_obb_pnnx.py',  'obb')

# # medium (m)
# patch_yolo_script('yolo11m_pnnx.py',      'det')
# patch_yolo_script('yolo11m_seg_pnnx.py',  'seg')
# patch_yolo_script('yolo11m_pose_pnnx.py', 'pose')
# patch_yolo_script('yolo11m_obb_pnnx.py',  'obb')


Patched yolo11n_pnnx.py successfully.


## 5. Re-export and Final Conversion
We now re-export the modified Python scripts back to TorchScript and perform the final PNNX conversion with dynamic input shapes.

In [4]:
import importlib

# ── Re-export patched scripts to .pt (direct import avoids Windows quoting issues) ──
# for mod_name in [
#     # nano (n)
#     'yolo11n_pnnx', 'yolo11n_seg_pnnx', 'yolo11n_pose_pnnx', 'yolo11n_obb_pnnx',
#     # small (s)
#     'yolo11s_pnnx', 'yolo11s_seg_pnnx', 'yolo11s_pose_pnnx', 'yolo11s_obb_pnnx',
#     # medium (m)
#     'yolo11m_pnnx', 'yolo11m_seg_pnnx', 'yolo11m_pose_pnnx', 'yolo11m_obb_pnnx',
# ]:
for mod_name in [
    # nano (n)
    'yolo11n_pnnx'
]:
    try:
        mod = importlib.import_module(mod_name)
        importlib.reload(mod)          # pick up any patches from cell 4
        mod.export_torchscript()
        print(f"Exported: {mod_name}.pt")
    except Exception as e:
        print(f"SKIP {mod_name}: {e}")

# ── Final PNNX conversion with dynamic shapes ─────────────────────────────────
# nano (n)
!pnnx yolo11n_pnnx.py.pt      inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11n_seg_pnnx.py.pt  inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11n_pose_pnnx.py.pt inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11n_obb_pnnx.py.pt  inputshape=[1,3,1024,1024] inputshape2=[1,3,512,512]
# # small (s)
# !pnnx yolo11s_pnnx.py.pt      inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11s_seg_pnnx.py.pt  inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11s_pose_pnnx.py.pt inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11s_obb_pnnx.py.pt  inputshape=[1,3,1024,1024] inputshape2=[1,3,512,512]
# # medium (m)
# !pnnx yolo11m_pnnx.py.pt      inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11m_seg_pnnx.py.pt  inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11m_pose_pnnx.py.pt inputshape=[1,3,640,640]   inputshape2=[1,3,320,320]
# !pnnx yolo11m_obb_pnnx.py.pt  inputshape=[1,3,1024,1024] inputshape2=[1,3,512,512]


SKIP yolo11n_pnnx: unexpected indent (yolo11n_pnnx.py, line 667)


pnnxparam = yolo11n_pnnx.py.pnnx.param
pnnxbin = yolo11n_pnnx.py.pnnx.bin
pnnxpy = yolo11n_pnnx.py_pnnx.py
pnnxonnx = yolo11n_pnnx.py.pnnx.onnx
ncnnparam = yolo11n_pnnx.py.ncnn.param
ncnnbin = yolo11n_pnnx.py.ncnn.bin
ncnnpy = yolo11n_pnnx.py_ncnn.py
fp16 = 1
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = [1,3,320,320]f32
customop = 
moduleop = 
open failed yolo11n_pnnx.py.pt
open failed yolo11n_pnnx.py.pt
open failed yolo11n_pnnx.py.pt
read_proto_from_binary failed
############# pass_level2


## 6. Organize Output Files
Rename the final `.param` and `.bin` files for cleaner usage.

In [6]:
import os
import shutil
from pathlib import Path

# ── 1. Rename pnnx outputs to clean names ──────────────────────────────────────
# pnnx names its outputs after the input file stem, so:
#   yolo11n_pnnx.py.pt  →  yolo11n_pnnx.py.ncnn.{param,bin}
# We rename those to  yolo11n.ncnn.{param,bin}, etc.
# cls models come out as yolo11n-cls.ncnn.{param,bin}  (hyphen) → yolo11n_cls.ncnn.{param,bin}

renames = [
    # (source produced by pnnx,              target clean name)
    # --- det / seg / pose / obb (dynamic-shape re-export path) ---
    ("yolo11n_pnnx.py.ncnn.param",      "yolo11n.ncnn.param"),
    ("yolo11n_pnnx.py.ncnn.bin",        "yolo11n.ncnn.bin"),
    # ("yolo11n_seg_pnnx.py.ncnn.param",  "yolo11n_seg.ncnn.param"),
    # ("yolo11n_seg_pnnx.py.ncnn.bin",    "yolo11n_seg.ncnn.bin"),
    # ("yolo11n_pose_pnnx.py.ncnn.param", "yolo11n_pose.ncnn.param"),
    # ("yolo11n_pose_pnnx.py.ncnn.bin",   "yolo11n_pose.ncnn.bin"),
    # ("yolo11n_obb_pnnx.py.ncnn.param",  "yolo11n_obb.ncnn.param"),
    # ("yolo11n_obb_pnnx.py.ncnn.bin",    "yolo11n_obb.ncnn.bin"),

    # ("yolo11s_pnnx.py.ncnn.param",      "yolo11s.ncnn.param"),
    # ("yolo11s_pnnx.py.ncnn.bin",        "yolo11s.ncnn.bin"),
    # ("yolo11s_seg_pnnx.py.ncnn.param",  "yolo11s_seg.ncnn.param"),
    # ("yolo11s_seg_pnnx.py.ncnn.bin",    "yolo11s_seg.ncnn.bin"),
    # ("yolo11s_pose_pnnx.py.ncnn.param", "yolo11s_pose.ncnn.param"),
    # ("yolo11s_pose_pnnx.py.ncnn.bin",   "yolo11s_pose.ncnn.bin"),
    # ("yolo11s_obb_pnnx.py.ncnn.param",  "yolo11s_obb.ncnn.param"),
    # ("yolo11s_obb_pnnx.py.ncnn.bin",    "yolo11s_obb.ncnn.bin"),

    # ("yolo11m_pnnx.py.ncnn.param",      "yolo11m.ncnn.param"),
    # ("yolo11m_pnnx.py.ncnn.bin",        "yolo11m.ncnn.bin"),
    # ("yolo11m_seg_pnnx.py.ncnn.param",  "yolo11m_seg.ncnn.param"),
    # ("yolo11m_seg_pnnx.py.ncnn.bin",    "yolo11m_seg.ncnn.bin"),
    # ("yolo11m_pose_pnnx.py.ncnn.param", "yolo11m_pose.ncnn.param"),
    # ("yolo11m_pose_pnnx.py.ncnn.bin",   "yolo11m_pose.ncnn.bin"),
    # ("yolo11m_obb_pnnx.py.ncnn.param",  "yolo11m_obb.ncnn.param"),
    # ("yolo11m_obb_pnnx.py.ncnn.bin",    "yolo11m_obb.ncnn.bin"),

    # # --- cls (static-shape path, pnnx uses the torchscript stem with hyphen) ---
    # ("yolo11n-cls.ncnn.param", "yolo11n_cls.ncnn.param"),
    # ("yolo11n-cls.ncnn.bin",   "yolo11n_cls.ncnn.bin"),
    # ("yolo11s-cls.ncnn.param", "yolo11s_cls.ncnn.param"),
    # ("yolo11s-cls.ncnn.bin",   "yolo11s_cls.ncnn.bin"),
    # ("yolo11m-cls.ncnn.param", "yolo11m_cls.ncnn.param"),
    # ("yolo11m-cls.ncnn.bin",   "yolo11m_cls.ncnn.bin"),
]

for src, dst in renames:
    if os.path.exists(src) and not os.path.exists(dst):
        os.rename(src, dst)
        print(f"Renamed: {src} → {dst}")
    elif os.path.exists(dst):
        print(f"Already exists: {dst}")
    else:
        print(f"WARNING – source not found: {src}")

# ── 2. Copy all NCNN assets to the Android project ─────────────────────────────
assets = Path(r"C:\YOLO-android\ncnn\ncnn-android-yolo11\app\src\main\assets")
assets.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "yolo11n.ncnn.bin",   "yolo11n.ncnn.param",
    # "yolo11n_cls.ncnn.bin", "yolo11n_cls.ncnn.param",
    # "yolo11n_seg.ncnn.bin", "yolo11n_seg.ncnn.param",
    # "yolo11n_pose.ncnn.bin","yolo11n_pose.ncnn.param",
    # "yolo11n_obb.ncnn.bin", "yolo11n_obb.ncnn.param",

    # "yolo11s.ncnn.bin",   "yolo11s.ncnn.param",
    # "yolo11s_cls.ncnn.bin", "yolo11s_cls.ncnn.param",
    # "yolo11s_seg.ncnn.bin", "yolo11s_seg.ncnn.param",
    # "yolo11s_pose.ncnn.bin","yolo11s_pose.ncnn.param",
    # "yolo11s_obb.ncnn.bin", "yolo11s_obb.ncnn.param",

    # "yolo11m.ncnn.bin",   "yolo11m.ncnn.param",
    # "yolo11m_cls.ncnn.bin", "yolo11m_cls.ncnn.param",
    # "yolo11m_seg.ncnn.bin", "yolo11m_seg.ncnn.param",
    # "yolo11m_pose.ncnn.bin","yolo11m_pose.ncnn.param",
    # "yolo11m_obb.ncnn.bin", "yolo11m_obb.ncnn.param",
]

copied, missing = [], []
for name in files_to_copy:
    src = Path(name)
    if src.exists():
        shutil.copy2(src, assets / name)
        copied.append(name)
    else:
        missing.append(name)

print(f"\nCopied {len(copied)} files → {assets}")
if missing:
    print("Missing (not copied):", missing)
else:
    print("All files copied successfully.")


Already exists: yolo11n.ncnn.param
Already exists: yolo11n.ncnn.bin

Copied 2 files → C:\YOLO-android\ncnn\ncnn-android-yolo11\app\src\main\assets
All files copied successfully.
